# 01 · Сбор данных

Этот ноутбук собирает сырые данные проекта.

- Часть 1: почасовой PM2.5 станции из OpenAQ (цепочка «локация → сенсор → измерения»).
- Часть 2: погода и CAMS из Open-Meteo (добавляется отдельно).

Запускать сверху вниз.

## Шаг 1. Импорты

Подключаем pandas и наш модуль `src/collect.py`. Ноутбук запускается из папки `notebooks/`, поэтому добавляем корень проекта в путь.

In [1]:
import os
import sys

import pandas as pd

sys.path.append("..")
from src import collect

RAW_PATH = "../data/raw/openaq_pm25.csv"

## Шаг 2. Что выгружаем

Сенсор PM2.5 и период взяты из задачи 1.1. Время в сырых данных — UTC, как его отдаёт API.

In [2]:
print("sensor_id:", collect.SENSOR_ID)
print("from (UTC):", collect.PERIOD_FROM_UTC)
print("to (UTC):", collect.PERIOD_TO_UTC)
start = pd.Timestamp(collect.PERIOD_FROM_UTC)
end = pd.Timestamp(collect.PERIOD_TO_UTC)
max_hours = int((end - start) / pd.Timedelta(hours=1))
print("max hours in period:", max_hours)

sensor_id: 25903
from (UTC): 2020-04-09T16:00:00Z
to (UTC): 2025-11-14T11:00:00Z
max hours in period: 49075


## Шаг 3. Выгрузка один раз

Качаем страницами по 1000 строк до пустой страницы, по годовым окнам. Между запросами пауза 1 с (лимит API — 60 запросов в минуту), при ответе 429 есть повтор. Сырой файл не перезаписываем: если он уже есть, API не вызываем.

In [3]:
if os.path.exists(RAW_PATH):
    print("Файл уже есть, API не вызываем:", RAW_PATH)
else:
    pm25_download = collect.download_pm25()
    collect.save_raw_csv(pm25_download, RAW_PATH)
    print("Сохранено строк:", len(pm25_download))

window: 2020-04-09T16:00:00Z .. 2021-01-01T00:00:00Z


page 1: 1000 rows


page 2: 1000 rows


page 3: 1000 rows


page 4: 1000 rows


page 5: 735 rows


... (38 outputs left out)

page 6: 1000 rows


page 7: 1000 rows


page 8: 574 rows


Сохранено строк: 37823


## Шаг 4. Читаем сырой файл

Читаем CSV с диска и смотрим размер и первые строки.

In [4]:
pm25_raw = pd.read_csv(RAW_PATH)
print(pm25_raw.shape)
pm25_raw.head()

(37823, 4)


,datetime_from_utc,datetime_to_utc,pm25,has_flags
0,2020-04-09T16:00:00Z,2020-04-09T17:00:00Z,6.0,False
1,2020-04-09T17:00:00Z,2020-04-09T18:00:00Z,10.0,False
2,2020-04-09T18:00:00Z,2020-04-09T19:00:00Z,8.0,False
3,2020-04-09T19:00:00Z,2020-04-09T20:00:00Z,6.0,False
4,2020-04-09T20:00:00Z,2020-04-09T21:00:00Z,4.0,False
